# Enveda v1 submission: FPNet single + 20ppm over 712K precomputed pool
No RDKit at inference. R1 NP-method.

In [ ]:
import sys
print(sys.version)
import torch
print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available())
assert torch.cuda.is_available(), 'need GPU'
import numpy as np, pyarrow.parquet as pq
print('numpy/pyarrow ok')

In [ ]:
from pathlib import Path
def find(name):
    hits = list(Path('/kaggle/input').rglob(name))
    assert hits, 'missing input: ' + name
    print(name, '->', hits[0])
    return str(hits[0])
TEST = find('test.parquet')
W_SINGLE = find('fp_single_s2.pt')
T_FP = find('fp.npy')
T_MASS = find('mass.npy')
T_KEYS = find('keys.npy')
T_SMI = find('smiles.txt')
C_FP = find('coco_fp.npy')
C_MASS = find('coco_mass.npy')
C_META = find('coco_meta.pkl')
print('torch cuda devices:', torch.cuda.device_count())

In [ ]:
import pickle, time
import numpy as np
t0=time.perf_counter()
t_fp = np.load(T_FP, mmap_mode='r')
t_mass = np.load(T_MASS, mmap_mode='r')
t_keys = np.load(T_KEYS, allow_pickle=True)
t_smi = open(T_SMI).read().splitlines()
print('train pool:', t_fp.shape, len(t_keys), len(t_smi), round(time.perf_counter()-t0,1))
assert t_fp.shape[0]==len(t_keys)==len(t_smi)==len(t_mass)==275810
meta = pickle.load(open(C_META,'rb'))
ckeys = list(meta['keys']); csmi = list(meta['smiles'])
c_mass = np.load(C_MASS, mmap_mode='r')
c_fp = np.load(C_FP, mmap_mode='r')
print('coco pool:', c_fp.shape, len(ckeys))
tset = set(t_keys)
cmap = dict(zip(ckeys, np.asarray(c_mass, float)))
cidx = {k:i for i,k in enumerate(ckeys)}
allk = list(t_keys) + [k for k in ckeys if k not in tset]
alls = t_smi + [s for k,s in zip(ckeys,csmi) if k not in tset]
allm = np.concatenate([np.asarray(t_mass, float), np.array([cmap[k] for k in ckeys if k not in tset], float)])
print('combined:', len(allk), round(time.perf_counter()-t0,1))
TQ = pq.read_table(TEST).to_pydict()
mol = {}
for i in range(len(TQ['molecule_id'])):
    mol.setdefault(TQ['molecule_id'][i], []).append(i)
print('test molecules:', len(mol), 'spectra:', len(TQ['molecule_id']))

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math
"""Spectrum -> molecular fingerprint model (CSI:FingerID-style neural ranker)."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math

MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+","[M+NH4]+","[M+Na]+","[M+K]+","[M-H2O+H]+","[M-2H2O+H]+","[M]+",
               "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+C2H4O2-H]-","[M+Cl]-","[M]-",
               "[M+2H]2+","[M-2H]-","[2M+H]+","[2M+Na]+","[2M+NH4]+","[2M-H]-","[2M+K]+",
               "[2M+CH2O2-H]-","[2M+C2H4O2-H]-","[2M+Na-2H]-","[M+Na-2H]-","[M-H2O]+","<unk>"]
ADDUCT_IX = {a:i for i,a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF","Orbitrap","QTOF","IT","other"]
INSTR_IX = {a:i for i,a in enumerate(INSTR_LIST)}

def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4

def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = it >= floor*mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        # keep the top `per_win` peaks inside each `win` Da bucket, then global top-N
        order = np.argsort(-it)
        bucket = (mz//win).astype(np.int64)
        cnt = {}; sel=[]
        for i in order:
            b = bucket[i]; c = cnt.get(b,0)
            if c < per_win: cnt[b]=c+1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks-len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it/it.max())
    return mz.astype(np.float32), v.astype(np.float32)

class SinEmb(nn.Module):
    """Log-spaced sinusoidal embedding for m/z values (Voronov et al.)."""
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim//2
        wav = torch.pow(10.0, (hi-lo)*torch.pow(torch.linspace(0,1,n), power) + lo)
        self.register_buffer('inv', (2*math.pi)/wav)
    def forward(self, x):                      # x: (...,)
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h=h
        self.n1=nn.LayerNorm(d); self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)
        self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d,d))
        self.drop=nn.Dropout(drop)
    def forward(self, x, pad):
        B,N,D=x.shape; y=self.n1(x)
        q,k,v = self.qkv(y).view(B,N,3,self.h,D//self.h).permute(2,0,3,1,4)
        m = (~pad)[:,None,None,:]                       # True = attend
        a = F.scaled_dot_product_attention(q,k,v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1,2).reshape(B,N,D)))
        return x + self.drop(self.ff(self.n2(x)))

class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d=d
        self.mz_emb  = SinEmb(d)
        self.nl_emb  = SinEmb(d)
        self.pk = nn.Linear(2*d+1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d+3, d)
        self.blocks = nn.ModuleList([Block(d,heads,drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2*d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))
    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B,N = mz.shape
        nl = (prec[:,None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce/100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1)/10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B,1,dtype=torch.bool,device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:,0]
        msk = (~pad[:,1:]).float().unsqueeze(-1)
        mean = (x[:,1:]*msk).sum(1)/msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))
ck = torch.load(W_SINGLE, map_location='cpu', weights_only=True)
net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).cuda().eval()
net.load_state_dict(ck['model'], strict=True)
print('model ok:', ck['nbits'], ck['d'], ck['layers'])
NTR = 275810
assert t_fp.shape == (275810, 867)

In [ ]:
MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
    S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
    Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {'[M+H]+':(1,1,PROTON),'[M+NH4]+':(1,1,NH4-E),
 '[M+Na]+':(1,1,MASS['Na']-E),'[M+K]+':(1,1,MASS['K']-E),
 '[M-H2O+H]+':(1,1,PROTON-H2O),'[M-2H2O+H]+':(1,1,PROTON-2*H2O),
 '[M+2H]2+':(1,2,2*PROTON),'[M]+':(1,1,-E),'[M-H2O]+':(1,1,-E-H2O),
 '[M+CH3OH+H]+':(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 '[M+CH3CN+H]+':(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 '[M-H]-':(1,1,-PROTON),'[M-H2O-H]-':(1,1,-PROTON-H2O),
 '[M+CH2O2-H]-':(1,1,FORMATE-PROTON),'[M+C2H4O2-H]-':(1,1,ACETATE-PROTON),
 '[M+Cl]-':(1,1,MASS['Cl']+E),'[M]-':(1,1,E),
 '[M-2H]-':(1,2,-2*PROTON),'[M+Na-2H]-':(1,1,MASS['Na']-2*PROTON),
 '[2M+H]+':(2,1,PROTON),'[2M+Na]+':(2,1,MASS['Na']-E),
 '[2M+NH4]+':(2,1,NH4-E),'[2M+K]+':(2,1,MASS['K']-E),
 '[2M-H]-':(2,1,-PROTON),'[2M+CH2O2-H]-':(2,1,FORMATE-PROTON),
 '[2M+C2H4O2-H]-':(2,1,ACETATE-PROTON),'[2M+Na-2H]-':(2,1,MASS['Na']-2*PROTON),
 '[3M+H]+':(3,1,PROTON),'[3M-H]-':(3,1,-PROTON)}
def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m = (ad == a)
        if m.any(): out[m] = (mz[m]*z - d)/n
    return out
print('mass contract ready')

In [ ]:
import numpy as np, torch, time
G = 4
WMAX = 2000
mids = list(mol.keys())
prows = []
for mid in mids:
    idxs = mol[mid]
    best = max(idxs, key=lambda i: len(TQ['ms2_mzs'][i]))
    prows.append((TQ['ms2_mzs'][best], TQ['ms2_normalized_intensities'][best], TQ['precursor_mz'][best],
            TQ['adduct'][best], TQ['instrument_type'][best], TQ['collision_energy_ev'][best], TQ['ionization_mode'][best]))
print('molecules:', len(mids))
@torch.no_grad()
def fwd_batch(rows):
    P = []
    for r in rows:
        a,b = prep_peaks(np.asarray(r[0],float), np.asarray(r[1],float), float(r[2]))
        P.append((a,b,r))
    Np = 128
    B = len(P)
    mz=np.zeros((B,Np),np.float32); it=np.zeros((B,Np),np.float32); pad=np.ones((B,Np),bool)
    for j,(a,b,_) in enumerate(P):
        n = min(len(a), Np)
        mz[j,:n]=a[:n]; it[j,:n]=b[:n]; pad[j,:n]=False
    T = lambda x: torch.as_tensor(x, device='cuda')
    rr = [r for _,_,r in P]
    args=(T(mz),T(it),T(pad),T(np.array([float(r[2]) for r in rr],np.float32)),
          T(np.array([ADDUCT_IX.get(r[3],ADDUCT_IX['<unk>']) for r in rr])),
          T(np.array([instr_family(r[4]) for r in rr])),
          T(np.array([float(np.mean(np.atleast_1d(r[5]))) if r[5] is not None else 25.0 for r in rr],np.float32)),
          T(np.array([1.0 if r[6]=='positive' else -1.0 for r in rr],np.float32)))
    return net(*args).float()
Z = torch.empty((len(prows), 6930))
BS = 64
for i in range(0, len(prows), BS):
    Z[i:i+BS] = fwd_batch(prows[i:i+BS]).cpu()
    if (i//BS+1) % 2 == 0: print('forward', min(i+BS, len(prows)), '/', len(prows))
print('all forwards done', tuple(Z.shape))
QM = neutral_mass(np.array([r[2] for r in prows], float), [r[3] for r in prows])
stat = {'empty_window':0,'mass_fallback':0,'cco':0,'exception':0}
print('forward+prep done', tuple(Z.shape))
NG = (len(mids) + G - 1)//G
print('groups:', NG, 'G=', G)
stat = {'empty_window':0,'mass_fallback':0,'cco':0,'exception':0}
PARTS = {}
def run_groups(g0, g1):
    import pickle
    tB = time.perf_counter()
    rows = []
    for g in range(g0*G, min(g1*G, len(mids)), G):
        try:
            chunk = range(g, min(g+G, len(mids)))
            Ws, KL, SL, PL = [], [], [], []
            for i in chunk:
                w = np.where(np.abs(allm - QM[i]) <= QM[i]*20e-6)[0]
                if len(w) > WMAX:
                    w = w[np.argsort(np.abs(allm[w]-QM[i]))[:WMAX]]
                Ws.append(w)
                KL.append([allk[j] for j in w])
                SL.append([alls[j] for j in w])
                PL.append([abs(allm[j]-QM[i])/QM[i]*1e6 for j in w])
            L = max([len(w) for w in Ws] + [1])
            Fmat = np.zeros((len(chunk), L, 6930), np.float32)
            Fmat_np = np.ones((len(chunk), L), bool)
            for bi, w in enumerate(Ws):
                is_tr = w < NTR
                if is_tr.any():
                    Fmat[bi,np.where(is_tr)[0]] = np.unpackbits(np.asarray(t_fp[w[is_tr]], dtype=np.uint8), axis=1)[:, :6930]
                if (~is_tr).any():
                    ci = np.array([cidx[allk[j]] for j in w[~is_tr]])
                    Fmat[bi,np.where(~is_tr)[0]] = np.unpackbits(np.asarray(c_fp[ci], dtype=np.uint8), axis=1)[:, :6930]
                Fmat_np[bi,:len(w)] = False
            Fg = torch.as_tensor(Fmat, device='cuda')
            Zg = torch.as_tensor(Z[list(chunk)].numpy(), device='cuda').unsqueeze(-1)
            with torch.no_grad():
                S = torch.bmm(Fg, Zg).squeeze(-1)
            S[Fmat_np] = float('-inf')
            order = torch.argsort(S, dim=1, descending=True).cpu().numpy()
            Scpu = S.cpu().numpy()
            for bi, i in enumerate(chunk):
                mid = mids[i]
                KLb, SLb, PLb = KL[bi], SL[bi], PL[bi]
                if not len(KLb):
                    stat['empty_window'] += 1
                    rows.append({'mid': mid, 'canon': [], 'smiles': [], 'base_scores': [], 'n_window': 0, 'status': 'empty_window'})
                    continue
                seen = set()
                canon_list, smiles_list, base_list = [], [], []
                for pos in order[bi]:
                    if pos >= len(KLb):
                        continue
                    k = KLb[pos]
                    if k in seen:
                        continue
                    seen.add(k)
                    canon_list.append(k)
                    smiles_list.append(SLb[pos])
                    base_list.append(float(Scpu[bi, pos]))
                    if len(canon_list) >= 50:
                        break
                if not canon_list:
                    stat['mass_fallback'] += 1
                    ranked = sorted(zip(KLb, SLb, PLb), key=lambda t: t[2])
                    for k, sm, pp in ranked[:50]:
                        if k in seen:
                            continue
                        seen.add(k)
                        canon_list.append(k)
                        smiles_list.append(sm)
                        base_list.append(float(-pp))
                    rows.append({'mid': mid, 'canon': canon_list, 'smiles': smiles_list, 'base_scores': base_list, 'n_window': len(KLb), 'status': 'mass_fallback'})
                    continue
                rows.append({'mid': mid, 'canon': canon_list, 'smiles': smiles_list, 'base_scores': base_list, 'n_window': len(KLb), 'status': 'ok'})
        except Exception as e:
            for i in chunk:
                rows.append({'mid': mids[i], 'canon': [], 'smiles': [], 'base_scores': [], 'n_window': 0, 'status': 'exception'})
            stat['exception'] += len(chunk)
        if (g//G+1) % 5 == 0: print('group', g//G+1, round(time.perf_counter()-tB,1), stat)
    PARTS[g0] = rows
    pickle.dump({'records': rows}, open(f'/kaggle/working/partial_{g0}.pkl','wb'))
    print('block done', g0, g1, 'rows', len(rows), round(time.perf_counter()-tB,1), stat)

In [ ]:
run_groups(0, 20)

In [ ]:
run_groups(20, 40)

In [ ]:
run_groups(40, 60)

In [ ]:
run_groups(60, 80)

In [ ]:
run_groups(80, 100)

In [ ]:
import subprocess, sys, os, glob, json, time
t0 = time.time()
def find_first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    print(pattern, '->', len(hits))
    return hits
GL_CANDS = find_first('/kaggle/input/**/gl_runner.py')
assert len(GL_CANDS) >= 1, 'gl_runner.py not found under /kaggle/input: ' + str(sorted(os.listdir('/kaggle/input')))
GL_PKG = os.path.dirname(GL_CANDS[0])
print('GL_PKG', GL_PKG)
assert os.path.exists(os.path.join(GL_PKG, 'gl_fuse.py')), 'gl_fuse.py missing'
assert os.path.exists(os.path.join(GL_PKG, 'ckpt', 'glacier.pt')), 'ckpt/glacier.pt missing'
WHL_FILES = find_first('/kaggle/input/**/*.whl')
print('whl files', len(WHL_FILES))
for p in WHL_FILES[:20]:
    print(p)
assert len(WHL_FILES) >= 9, f'wheelhouse incomplete: {len(WHL_FILES)}'
WHEEL_DIR = os.path.dirname(WHL_FILES[0])
print('WHEEL_DIR', WHEEL_DIR, sorted(os.listdir(WHEEL_DIR)))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps', '--find-links', WHEEL_DIR,
    'rdkit', 'einops', 'pytorch_lightning', 'lightning-utilities', 'torchmetrics', 'fsspec', 'typing_extensions', 'packaging', 'PyYAML'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-2000:])
import importlib
for m in ('rdkit', 'einops', 'pytorch_lightning', 'torchmetrics'):
    v = importlib.import_module(m)
    print(m, getattr(v, '__version__', 'ok'))
all_records = []
for g0 in sorted(PARTS):
    all_records.extend(PARTS[g0])
print('assembled records', len(all_records), 'from blocks', sorted(PARTS))
rec_by_mid = {r['mid']: r for r in all_records}
assert len(rec_by_mid) == len(mids), f'records {len(rec_by_mid)} != mids {len(mids)}'
n_top50 = sum(len(r['smiles']) for r in all_records)
print('total top50 cands', n_top50, 'empty', sum(1 for r in all_records if not r['smiles']))
items = []
for mid in mids:
    rec = rec_by_mid[mid]
    cands = list(rec['smiles'][:50])
    idxs = mol[mid]
    specs = []
    for ii in idxs:
        ce_ev = TQ['collision_energy_ev'][ii]
        specs.append({
            'mz': list(map(float, TQ['ms2_mzs'][ii])),
            'it': list(map(float, TQ['ms2_normalized_intensities'][ii])),
            'prec': float(TQ['precursor_mz'][ii]),
            'adduct': str(TQ['adduct'][ii]),
            'mode': str(TQ['ionization_mode'][ii]),
            'instrument': str(TQ['instrument_type'][ii]),
            'ce_ev': list(map(float, ce_ev)) if ce_ev is not None else [],
            'ce_orig': str(TQ['collision_energy_orig'][ii]) if TQ['collision_energy_orig'][ii] is not None else '',
            'ce_units': str(TQ['collision_energy_orig_units'][ii]) if TQ['collision_energy_orig_units'][ii] is not None else '',
        })
    items.append({'mid': mid, 'cands': cands, 'spectra': specs})
print('items', len(items), 'total cands', sum(len(x['cands']) for x in items), 'total spectra', sum(len(x['spectra']) for x in items))
sys.path.insert(0, GL_PKG)
from gl_fuse import run_gl
t1 = time.time()
out = run_gl(GL_PKG, items, workdir='/kaggle/working/gl_work', device='cuda', budget_s=2400.0, site='none', wheels=None, extra_args=('--any-rdkit',), log=print)
print(f'run_gl returned in {time.time()-t1:.1f}s, n_mols {len(out)}')
assert isinstance(out, dict) and len(out) == len(mids), f'gl output incomplete: {len(out) if isinstance(out, dict) else type(out)}'
with open('/kaggle/working/glacier_scores.json', 'w') as f:
    json.dump(out, f)
print('wrote /kaggle/working/glacier_scores.json', round(os.path.getsize('/kaggle/working/glacier_scores.json')/1024, 1), 'KB')
import shutil
meta_src = '/kaggle/working/gl_work/gl_out.json.meta.json'
assert os.path.exists(meta_src), f'runner meta missing: {glob.glob("/kaggle/working/gl_work/*")}'
shutil.copyfile(meta_src, '/kaggle/working/gl_meta.json')
meta = json.load(open(meta_src))
print(json.dumps({k: meta.get(k) for k in ('status','n_mols','n_mols_covered','n_mols_scored','n_cands','n_scored','seconds','pred_per_s','device','rdkit','max_mem_gb','ckpt_sha256','ms_pred_commit')}, indent=1))

In [ ]:
import math, json, csv, os, pickle
def _z(vals):
    ok = [v is not None and isinstance(v, (int, float)) and math.isfinite(v) for v in vals]
    xs = [float(v) for v, o in zip(vals, ok) if o]
    n = len(xs)
    if n < 2:
        return [0.0] * len(vals)
    mu = sum(xs) / n
    sd = math.sqrt(sum((x - mu) ** 2 for x in xs) / (n - 1))
    if not sd > 0:
        return [0.0] * len(vals)
    return [((float(v) - mu) / sd) if o else 0.0 for v, o in zip(vals, ok)]
try:
    import rdkit
    print('rdkit =', rdkit.__version__)
    from rdkit import Chem
    HAVE_RDKIT = True
except Exception as e:
    print('rdkit import failed', repr(e)[:500])
    HAVE_RDKIT = False
    Chem = None
all_records = []
for g0 in sorted(PARTS):
    all_records.extend(PARTS[g0])
print('fuse assembled', len(all_records), 'from blocks', sorted(PARTS))
rec_by_mid = {r['mid']: r for r in all_records}
assert len(rec_by_mid) == len(mids)
gl_out = json.load(open('/kaggle/working/glacier_scores.json'))
assert isinstance(gl_out, dict) and len(gl_out) == len(mids), f'glacier_scores incomplete: {len(gl_out)}'
out_rows = []
stat_fuse = {'cco_empty': 0, 'pad_cco': 0, 'raw_fallback': 0, 'raw_fallback_no_rdkit': 0, 'n_scored_mols': 0, 'n_scored_cands': 0, 'n_total_cands': 0}
for mid in mids:
    rec = rec_by_mid[mid]
    smiles = list(rec['smiles'])
    base = list(rec['base_scores'])
    n = len(smiles)
    stat_fuse['n_total_cands'] += n
    if n == 0:
        out_rows.append((mid, ';'.join(['CCO']*25)))
        stat_fuse['cco_empty'] += 1
        continue
    gdict = gl_out.get(mid, {})
    gl = [gdict.get(sm) for sm in smiles]
    n_sc = sum(1 for v in gl if v is not None and isinstance(v, (int, float)) and math.isfinite(float(v)))
    stat_fuse['n_scored_cands'] += n_sc
    if n_sc > 0:
        stat_fuse['n_scored_mols'] += 1
    zb = _z(base)
    zg = _z(gl)
    fused = [a + b for a, b in zip(zb, zg)]
    scored_idx = [i for i, v in enumerate(gl) if v is not None and isinstance(v, (int, float)) and math.isfinite(float(v))]
    sset = set(scored_idx)
    unscored_idx = [i for i in range(n) if i not in sset]
    scored_sorted = sorted(scored_idx, key=lambda i: (-fused[i], i))
    unscored_sorted = sorted(unscored_idx, key=lambda i: (-base[i], i))
    full_order = scored_sorted + unscored_sorted
    top25_idx = full_order[:25]
    top25_raw = [smiles[i] for i in top25_idx]
    if len(top25_raw) < 25:
        top25_raw = top25_raw + ['CCO'] * (25 - len(top25_raw))
        stat_fuse['pad_cco'] += 1
    canon_out = []
    for sm in top25_raw:
        if sm == 'CCO':
            canon_out.append('CCO')
            continue
        if not HAVE_RDKIT:
            canon_out.append(sm)
            stat_fuse['raw_fallback_no_rdkit'] += 1
            continue
        try:
            m = Chem.MolFromSmiles(sm)
            if m is None:
                canon_out.append(sm)
                stat_fuse['raw_fallback'] += 1
            else:
                canon_out.append(Chem.MolToSmiles(m))
        except Exception:
            canon_out.append(sm)
            stat_fuse['raw_fallback'] += 1
    out_rows.append((mid, ';'.join(canon_out)))
print('fused rows', len(out_rows), stat_fuse)
print('blocks stat', stat)
with open('/kaggle/working/submission.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['molecule_id','smiles'])
    w.writerows(out_rows)
ncco_first = sum(1 for _, s in out_rows if s.split(';')[0] == 'CCO')
print('wrote', len(out_rows), 'bare_CCO_first', ncco_first)
print('coverage blocks', stat, 'fuse', stat_fuse)
assert len(out_rows)==len(mids)
for _, s in out_rows:
    assert len(s.split(';'))==25, 'row must have 25 cands'
print('verified 400x25' if len(out_rows)==400 else f'verified {len(out_rows)}x25')

产物 `/kaggle/working/submission.csv`（v8：FPNet top50 + GLACIER z-融合 top25，RDKit 规范导出）。覆盖率随行打印；CCO 仅空窗/异常/补位时出现；RDKit 版本如实打印。